# Session 2 
### Iteration 2 - Embedding Model Evaluation

#### New concepts for this iteration:
 - Introduced two new embedding models (in addition to the default ChromaDB model)
 - Create test queries to evaluate the quality of the different embedding models (and our chunking configuration) on results.  List out the results of the different models to compare quality of results. 
 -
#### Compare three embedding models on the Baldwin Boro regulation PDFs

| Model | Type | Cost |
|---|---|---|
| `all-MiniLM-L6-v2` | ChromaDB default (local) | Free |
| `all-mpnet-base-v2` | HuggingFace sentence-transformers (local) | Free |
| `text-embedding-3-small` | OpenAI API | ~$0.02 / 1M tokens |

Each model gets its own ChromaDB collection. The same queries are run against all three and results are printed side-by-side.


In [1]:
import os
import re
import chromadb
import pypdf
import tiktoken
import huggingface_hub
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from chromadb.utils.embedding_functions import (
    DefaultEmbeddingFunction,
    SentenceTransformerEmbeddingFunction,
    OpenAIEmbeddingFunction,
)

# Load API key for OpenAI and HuggingFace
load_dotenv(find_dotenv())

try:
    huggingface_hub.login(token=os.environ.get("HUGGINGFACE_API_KEY"), add_to_git_credential=False)
    print("HuggingFace: authenticated")
except Exception as e:
    print(f"HuggingFace: proceeding without auth ({e})")

# This regex pattern is used to extract sections from the PDF files.  Asssume that this will
# need to be adjusted for different PDF files.  Also assume that the PDF files are well formed. 
# Assume that chunking by section is the best way to go.
SECTION_PATTERN = re.compile(
    r'(?=(?:Chapter\s+[A-Z]?\d+|ARTICLE\s+[IVXLC]+|§\s*[A-Z]?\d+[-\w]*\.))',
    re.MULTILINE
)
OVERLAP_CHARS = 200

# This function extracts sections from a PDF file.  It uses the SECTION_PATTERN to extract sections.
def extract_sections(pdf_path: Path) -> list[dict]:
    reader = pypdf.PdfReader(pdf_path)
    full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
    raw_chunks = SECTION_PATTERN.split(full_text)
    raw_chunks = [c.strip() for c in raw_chunks if c.strip()]

    sections = []
    for i, chunk in enumerate(raw_chunks):
        overlap = raw_chunks[i - 1][-OVERLAP_CHARS:] if i > 0 else ""
        heading = chunk.splitlines()[0].strip()
        sections.append({
            "text": overlap + chunk,
            "source": pdf_path.name,
            "chunk_index": i,
            "heading": heading,
        })
    return sections

HuggingFace: proceeding without auth (Invalid user token.)


In [2]:
# This code loads the PDF files from the data directory and extracts the sections.
data_dir = Path("data")
all_sections = []
for pdf_path in sorted(data_dir.glob("*.pdf")):
    all_sections.extend(extract_sections(pdf_path))

print(f"Loaded {len(all_sections)} chunks from {len(list(data_dir.glob('*.pdf')))} PDFs")

Loaded 2061 chunks from 3 PDFs


In [3]:
import openai as openai_client

OPENAI_MAX_TOKENS = 7_500  # safely under the 8191 token limit for text-embedding-3-small
OPENAI_BATCH_SIZE = 500    # stay well under OpenAI's 2048 inputs-per-request limit
_enc = tiktoken.get_encoding("cl100k_base")

# This function sanitizes the text by removing any null bytes and encoding it to UTF-8.
# It also limits the number of tokens to the maximum number of tokens allowed by the model.
def sanitize(text: str, max_tokens: int | None = None) -> str:
    text = text.replace("\x00", "").encode("utf-8", errors="ignore").decode("utf-8").strip()
    if max_tokens:
        tokens = _enc.encode(text)
        if len(tokens) > max_tokens:
            text = _enc.decode(tokens[:max_tokens])
    return text

# This function embeds the text using the OpenAI API.
# It embeds the text in batches of 500 to stay under the 2048 inputs-per-request limit.
def embed_openai_batched(docs: list[str], model: str = "text-embedding-3-small") -> list[list[float]]:
    client_oa = openai_client.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    embeddings = []
    for i in range(0, len(docs), OPENAI_BATCH_SIZE):
        batch = docs[i:i + OPENAI_BATCH_SIZE]
        response = client_oa.embeddings.create(input=batch, model=model)
        embeddings.extend([r.embedding for r in response.data])
        print(f"  OpenAI: embedded {min(i + OPENAI_BATCH_SIZE, len(docs))}/{len(docs)} docs")
    return embeddings

chroma_client = chromadb.EphemeralClient()
collections = {}

# We will build three collections:  one for each model.
# --- Store embeddings for local models:  all-MiniLM-L6-v2 and all-mpnet-base-v2 ---
for model_name, ef in [
    ("minilm (default)", DefaultEmbeddingFunction()),
    ("mpnet (huggingface)", SentenceTransformerEmbeddingFunction(model_name="all-mpnet-base-v2")),
]:
    docs = [sanitize(s["text"]) for s in all_sections]
    col = chroma_client.get_or_create_collection(
        name=model_name.replace(" ", "_").replace("(", "").replace(")", ""),
        embedding_function=ef,
    )
    col.add(
        documents=docs,
        metadatas=[{"source": s["source"], "chunk_index": s["chunk_index"], "heading": s["heading"]} for s in all_sections],
        ids=[f"{s['source']}_chunk_{s['chunk_index']}" for s in all_sections],
    )
    collections[model_name] = col
    print(f"Built collection: {model_name}")

# --- OpenAI model (pre-computed embeddings in controlled batches) ---
openai_docs = [sanitize(s["text"], OPENAI_MAX_TOKENS) for s in all_sections]
print("Building OpenAI embeddings...")
openai_embeddings = embed_openai_batched(openai_docs)
col = chroma_client.get_or_create_collection(
    name="text_embedding_3_small_openai",
    embedding_function=OpenAIEmbeddingFunction(api_key=os.environ["OPENAI_API_KEY"], model_name="text-embedding-3-small"),
)
col.add(
    documents=openai_docs,
    embeddings=openai_embeddings,
    metadatas=[{"source": s["source"], "chunk_index": s["chunk_index"], "heading": s["heading"]} for s in all_sections],
    ids=[f"{s['source']}_chunk_{s['chunk_index']}" for s in all_sections],
)
collections["text-embedding-3-small (openai)"] = col
print("Built collection: text-embedding-3-small (openai)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Built collection: minilm (default)
Built collection: mpnet (huggingface)
Building OpenAI embeddings...
  OpenAI: embedded 500/2061 docs
  OpenAI: embedded 1000/2061 docs
  OpenAI: embedded 1500/2061 docs
  OpenAI: embedded 2000/2061 docs
  OpenAI: embedded 2061/2061 docs
Built collection: text-embedding-3-small (openai)


In [ ]:
SNIPPET_CHARS = 1000

# This function compares the results of the three models.
# It takes a query and the number of results to return.
# It then prints the results side-by-side for the three models.
def compare(query: str, n_results: int = 5):
    print(f"\nQUERY: {query}\n{'=' * 80}")
    for model_name, col in collections.items():
        results = col.query(query_texts=[query], n_results=n_results)
        print(f"\n--- {model_name.upper()} ---")
        for text, meta, distance in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ):
            snippet = " ".join(text.split())[:SNIPPET_CHARS]
            print(f"  Score: {1 - distance:.3f} | {meta['source']} — {meta['heading']}")
            print(f"    {snippet}...")

compare("do i have to register with the boro if i want to install an alarm?")
compare("what are the rules for parking on residential streets?")
compare("how do i appeal a zoning decision?")
compare("What company is authorized to provide and maintain the telephone service within the boro?")
compare("I'd like to put an alarm system in my business in the borough.  What are the requirements to do this?")
compare("Can I put solar panels on my house roof?  What are the steps for approval?") 
compare("I'm a police officer and would like to work for the borough until I'm 67 years old.  Is this possible?")
compare("I have a construction company, how can I be included on the list of qualified suppliers for borough contracts? ")
compare("What is the overtime policy for the borough?")
compare("What paid holidays does the borough observe?")
compare("what is the leaf removal policy for the borough?")
compare("Where can I find information about the borough's recycling program?")
compare("Where can I find information on the borough's building codes for residential construction?")
compare("What are the requirements for a business license in the borough?")
compare("what does the code enforcement officier do")
compare("What does the building inspector do?")
compare("Can I have a fire pit in my backyard?")    







QUERY: do i have to register with the boro if i want to install an alarm?

--- MINILM (DEFAULT) ---
  Score: 0.142 | BaldwinBoro-GeneralLegislation.pdf — §  62-6. Duties of alarm user.
    assified as use of a nonregistered alarm system, and citations and penalties shall be assessed without waiver. A twenty-five-dollar late fee can be assessed if the renewal is more than 30 days late.§ 62-6. Duties of alarm user. E. Any false statement of a material fact made by an applicant for the p...
  Score: 0.089 | BaldwinBoro-GeneralLegislation.pdf — §  62-7. Duties of alarm company; permit required; fee.
    function in the alarm system to be more false alarm resistant or provide additional user Borough of Baldwin, PA § 62-3 ALARM SYSTEMS § 62-6 Downloaded from https://ecode360.com/BA0968 on 2026-06-17§ 62-7. Duties of alarm company; permit required; fee. training as appropriate. See Appendix A for Inst...
  Score: 0.079 | BaldwinBoro-GeneralLegislation.pdf — §  62-7. Duties of alarm company;
